# Дополнительная практика: от очищенных данных к аналитическому мини-отчёту

В этом notebook мы продолжаем основной кейс. Данные уже знакомы: продажи, товары, регионы и клиенты. Теперь задача — собрать компактный аналитический результат: KPI, таблицы, графики, SQL-сверку и короткий вывод.

Рабочая цепочка:

```text
подготовленные данные → KPI → аналитические срезы → графики → SQL-сверка → мини-отчёт
```

## 1. Подготовка среды

Запускайте ячейки сверху вниз. Все пути в notebook относительные, поэтому важно открыть проект из корневой папки.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'raw').exists() and (PROJECT_ROOT.parent / 'data' / 'raw').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
SQL_DIR = PROJECT_ROOT / 'sql'

OUTPUT_DIR.mkdir(exist_ok=True)

print('Корень проекта:', PROJECT_ROOT)
print('Папка данных:', RAW_DIR)
print('Папка результатов:', OUTPUT_DIR)

## Русские подписи для таблиц

В коде ниже мы создаём словарь русских названий столбцов и функцию `display_ru()`. Она нужна только для удобного чтения результата. Исходные таблицы и переменные не переименовываются, поэтому код и SQL-запросы продолжают работать как раньше.


In [ ]:
RUSSIAN_COLUMNS = {
    "order_id": "Номер заказа",
    "order_date": "Дата заказа",
    "client_id": "Код клиента",
    "product_id": "Код товара",
    "region_id": "Код региона",
    "channel": "Канал продаж",
    "quantity": "Количество",
    "unit_price": "Цена за единицу",
    "discount": "Скидка",
    "revenue": "Выручка",
    "order_month": "Месяц заказа",
    "is_discounted": "Есть скидка",
    "product_name": "Название товара",
    "category": "Категория",
    "cost": "Себестоимость",
    "region_name": "Регион",
    "macro_region": "Макрорегион",
    "segment": "Сегмент клиента",
    "registration_date": "Дата регистрации",
    "loyalty_level": "Уровень лояльности",
    "orders_count": "Количество заказов",
    "clients_count": "Количество клиентов",
    "total_quantity": "Общее количество",
    "total_revenue": "Общая выручка",
    "avg_order_revenue": "Средняя выручка на заказ",
    "avg_discount": "Средняя скидка",
    "discount_group": "Группа скидки",
    "revenue_diff": "Разница в выручке",
    "orders_diff": "Разница в заказах",
    "problem": "Проблема",
    "rows_count": "Количество строк",
}

RUSSIAN_PROBLEMS = {
    "orders_without_product": "заказы без найденного товара",
    "orders_without_region": "заказы без найденного региона",
    "orders_without_client": "заказы без найденного клиента",
}

def display_ru(df, rows=10):
    """Показать таблицу с русскими подписями столбцов, не меняя исходный DataFrame."""
    ru_df = df.copy()
    if "problem" in ru_df.columns:
        ru_df["problem"] = ru_df["problem"].replace(RUSSIAN_PROBLEMS)
    ru_df = ru_df.rename(columns=RUSSIAN_COLUMNS)
    display(ru_df.head(rows))
    return ru_df

print('Русские подписи для таблиц подключены.')


In [ ]:
required_files = [
    RAW_DIR / 'sales.csv',
    RAW_DIR / 'products.xlsx',
    RAW_DIR / 'regions.json',
    RAW_DIR / 'clients.csv',
    SQL_DIR / 'analytics_demo.sqlite',
]

for file_path in required_files:
    if file_path.exists():
        print('OK:', file_path)
    else:
        print('НЕ НАЙДЕН:', file_path)

## 2. Загрузка и быстрая подготовка данных

Здесь мы повторяем базовую подготовку без подробного разбора. Главная цель — получить таблицу `sales_full`, с которой дальше будем строить мини-отчёт.

In [ ]:
sales_raw = pd.read_csv(RAW_DIR / 'sales.csv')
products_raw = pd.read_excel(RAW_DIR / 'products.xlsx')
regions_raw = pd.read_json(RAW_DIR / 'regions.json')
clients_raw = pd.read_csv(RAW_DIR / 'clients.csv')

print('sales_raw:', sales_raw.shape)
print('products_raw:', products_raw.shape)
print('regions_raw:', regions_raw.shape)
print('clients_raw:', clients_raw.shape)

In [ ]:
sales = sales_raw.copy()
products = products_raw.copy()
regions = regions_raw.copy()
clients = clients_raw.copy()

sales['order_date'] = pd.to_datetime(sales['order_date'], errors='coerce')
sales['channel'] = sales['channel'].astype('string').str.strip().str.lower()
products['category'] = products['category'].astype('string').str.strip().str.lower()
clients['segment'] = clients['segment'].astype('string').str.strip()
clients['loyalty_level'] = clients['loyalty_level'].astype('string').str.strip().str.title()

sales = sales.drop_duplicates(subset=['order_id'], keep='first')
sales = sales[
    (sales['quantity'] > 0) &
    (sales['unit_price'] > 0) &
    (sales['discount'].between(0, 1)) &
    (sales['order_date'].notna())
].copy()

sales['revenue'] = sales['quantity'] * sales['unit_price'] * (1 - sales['discount'])
sales['order_month'] = sales['order_date'].dt.to_period('M').astype(str)
sales['is_discounted'] = sales['discount'] > 0
sales['discount_group'] = pd.cut(
    sales['discount'],
    bins=[-0.001, 0, 0.10, 0.25, 1.0],
    labels=['без скидки', 'скидка до 10%', 'скидка 10-25%', 'скидка выше 25%'],
    include_lowest=True
)

sales_full = (
    sales
    .merge(products, on='product_id', how='left')
    .merge(regions, on='region_id', how='left')
    .merge(clients, on='client_id', how='left')
)

print('Строк после подготовки:', len(sales_full))
display(sales_full.head())

## 3. KPI-витрина

KPI-витрина — это компактная таблица с главными показателями. Она нужна, чтобы быстро понять масштаб данных и общий результат.

In [ ]:
kpi_summary = pd.DataFrame({
    'metric': [
        'total_revenue',
        'orders_count',
        'avg_order_revenue',
        'avg_discount',
        'discounted_orders_share',
        'rows_after_cleaning'
    ],
    'value': [
        round(float(sales_full['revenue'].sum()), 2),
        int(sales_full['order_id'].nunique()),
        round(float(sales_full['revenue'].mean()), 2),
        round(float(sales_full['discount'].mean()), 4),
        round(float(sales_full['is_discounted'].mean()), 4),
        int(len(sales_full))
    ]
})

display(kpi_summary)

In [ ]:
display_ru(kpi_summary) # KPI с русскими подписями

## 4. Аналитические срезы

Сделаем несколько таблиц: динамика по месяцам, топ-5 регионов, категории и группы скидок.

In [ ]:
monthly_revenue = (
    sales_full
    .groupby('order_month', as_index=False)
    .agg(
        orders_count=('order_id', 'nunique'),
        total_revenue=('revenue', 'sum'),
        avg_order_revenue=('revenue', 'mean')
    )
    .sort_values('order_month')
)
monthly_revenue[['total_revenue', 'avg_order_revenue']] = monthly_revenue[['total_revenue', 'avg_order_revenue']].round(2)
display(monthly_revenue)

In [ ]:
display_ru(monthly_revenue) # динамика с русскими подписями

In [ ]:
top_regions = (
    sales_full
    .groupby('region_name', dropna=False, as_index=False)
    .agg(
        orders_count=('order_id', 'nunique'),
        total_revenue=('revenue', 'sum'),
        avg_order_revenue=('revenue', 'mean')
    )
    .sort_values('total_revenue', ascending=False)
    .head(5)
)
top_regions[['total_revenue', 'avg_order_revenue']] = top_regions[['total_revenue', 'avg_order_revenue']].round(2)
display(top_regions)

In [ ]:
display_ru(top_regions) # регионы с русскими подписями

In [ ]:
top_categories = (
    sales_full
    .groupby('category', dropna=False, as_index=False)
    .agg(
        orders_count=('order_id', 'nunique'),
        total_revenue=('revenue', 'sum'),
        avg_order_revenue=('revenue', 'mean')
    )
    .sort_values('total_revenue', ascending=False)
)
top_categories[['total_revenue', 'avg_order_revenue']] = top_categories[['total_revenue', 'avg_order_revenue']].round(2)
display(top_categories)

In [ ]:
display_ru(top_categories) # категории с русскими подписями

In [ ]:
discount_group_summary = (
    sales_full
    .groupby('discount_group', observed=False, as_index=False)
    .agg(
        orders_count=('order_id', 'nunique'),
        total_revenue=('revenue', 'sum'),
        avg_order_revenue=('revenue', 'mean')
    )
    .sort_values('total_revenue', ascending=False)
)
discount_group_summary[['total_revenue', 'avg_order_revenue']] = discount_group_summary[['total_revenue', 'avg_order_revenue']].round(2)
display(discount_group_summary)

In [ ]:
display_ru(discount_group_summary) # группы скидок с русскими подписями

In [ ]:
category_channel_matrix = pd.pivot_table(
    sales_full,
    index='category',
    columns='channel',
    values='revenue',
    aggfunc='sum',
    fill_value=0
).round(2)

display(category_channel_matrix)

## 5. Визуализация

Построим три простых графика. Важно не только построить график, но и понять, какой вопрос он закрывает.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly_revenue['order_month'], monthly_revenue['total_revenue'], marker='o')
plt.title('Динамика выручки по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Выручка')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_regions['region_name'].astype(str), top_regions['total_revenue'])
plt.title('Топ-5 регионов по выручке')
plt.xlabel('Регион')
plt.ylabel('Выручка')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_categories['category'].astype(str), top_categories['total_revenue'])
plt.title('Выручка по категориям')
plt.xlabel('Категория')
plt.ylabel('Выручка')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. SQL-сверка

Теперь повторим часть логики через SQLite. Это нужно, чтобы проверить, что аналитическая логика не зависит только от pandas.

In [ ]:
connection = sqlite3.connect(SQL_DIR / 'analytics_demo.sqlite')

query = '''
WITH sales_enriched AS (
    SELECT
        order_id,
        quantity,
        unit_price,
        discount,
        quantity * unit_price * (1 - discount) AS revenue,
        CASE
            WHEN discount = 0 THEN 'без скидки'
            WHEN discount <= 0.10 THEN 'скидка до 10%'
            WHEN discount <= 0.25 THEN 'скидка 10-25%'
            ELSE 'скидка выше 25%'
        END AS discount_group
    FROM sales
    WHERE quantity > 0
      AND unit_price > 0
      AND discount BETWEEN 0 AND 1
)
SELECT
    discount_group,
    COUNT(DISTINCT order_id) AS orders_count,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(AVG(revenue), 2) AS avg_order_revenue
FROM sales_enriched
GROUP BY discount_group
ORDER BY total_revenue DESC;
'''

sql_discount_group_summary = pd.read_sql_query(query, connection)
display(sql_discount_group_summary)

In [ ]:
connection.close()
print('Соединение с SQLite закрыто.')

## 7. Сохранение результатов

Сохраняем итоговые таблицы, чтобы их можно было передать или приложить к отчёту.

In [ ]:
kpi_summary.to_csv(OUTPUT_DIR / 'extra_kpi_summary.csv', index=False)
monthly_revenue.to_csv(OUTPUT_DIR / 'extra_monthly_revenue.csv', index=False)
top_regions.to_csv(OUTPUT_DIR / 'extra_top_regions.csv', index=False)
top_categories.to_csv(OUTPUT_DIR / 'extra_top_categories.csv', index=False)
discount_group_summary.to_csv(OUTPUT_DIR / 'extra_discount_group_summary.csv', index=False)
category_channel_matrix.to_excel(OUTPUT_DIR / 'extra_category_channel_matrix.xlsx')

print('Файлы сохранены:')
for file_path in sorted(OUTPUT_DIR.glob('extra_*')):
    print(file_path)

## 8. Мини-отчёт

Сформулируйте 5–6 тезисов по структуре:

1. Что анализировалось.
2. Какие данные были подготовлены.
3. Какие показатели рассчитаны.
4. Что видно по регионам, категориям, каналам или скидкам.
5. Какие ограничения качества данных нужно учитывать.
6. Какой следующий шаг стоит сделать.

Напишите вывод ниже в markdown-ячейке.

### Подсказка по русскому мини-отчёту

В мини-отчёте пишите не названия переменных, а русские управленческие формулировки: «выручка выросла/снизилась», «лидирует регион», «категория даёт основной вклад», «скидки влияют на средний чек», «качество данных ограничивает вывод». 
